In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
def listar_arquivos(pasta: str):
    if not os.path.exists(pasta):
        print(f"A pasta '{pasta}' não existe.")
        return

    arquivos = [f for f in os.listdir(pasta) if os.path.isfile(os.path.join(pasta, f))]
    lista_arquivos = []
    for arquivo in arquivos:
        lista_arquivos.append(arquivo)

    if lista_arquivos:
        return [arquivo for arquivo in lista_arquivos if 'landmarks' in arquivo]
    else:
        print(f"Nenhum arquivo encontrado na pasta '{pasta}'.")

In [3]:
# ---------- utilidades geométricas ----------
def angle_deg(a, b, c):
    """
    Ângulo ABC (em graus), dado A,B,C como arrays (N,2) ou (N,3).
    Retorna array (N,).
    """
    ba = a - b
    bc = c - b
    # produto escalar e norma
    dot = np.sum(ba * bc, axis=1)
    n1 = np.linalg.norm(ba, axis=1)
    n2 = np.linalg.norm(bc, axis=1)
    denom = (n1 * n2) + 1e-9
    cosang = np.clip(dot / denom, -1.0, 1.0)
    return np.degrees(np.arccos(cosang))


def rolling_smooth(x, win=7):
    # win ímpar é melhor
    win = max(3, int(win))
    if win % 2 == 0:
        win += 1
    return pd.Series(x).rolling(win, center=True, min_periods=1).mean().to_numpy()

In [4]:
# ---------- detecção simples de picos/vales (sem SciPy) ----------
def local_extrema(x, order=5, mode="max"):
    """
    Retorna índices de máximos/minimos locais.
    order = tamanho da vizinhança: o ponto deve ser o maior/menor dentro do intervalo [i-order, i+order]
    """
    x = np.asarray(x)
    idx = []
    n = len(x)
    for i in range(order, n - order):
        window = x[i - order : i + order + 1]
        if mode == "max":
            if x[i] == np.max(window):
                idx.append(i)
        else:
            if x[i] == np.min(window):
                idx.append(i)
    return np.array(idx, dtype=int)


def filter_by_distance(indices, min_distance):
    """
    Se tiver eventos muito próximos, mantém só um a cada min_distance (o primeiro).
    """
    if len(indices) == 0:
        return indices
    kept = [indices[0]]
    last = indices[0]
    for i in indices[1:]:
        if i - last >= min_distance:
            kept.append(i)
            last = i
    return np.array(kept, dtype=int)


def filter_by_prominence(x, indices, prominence=5.0, mode="max"):
    """
    Filtro simples de 'prominence': compara com vizinhos imediatos.
    Não é o prominence verdadeiro do SciPy, mas ajuda a cortar tremulações.
    """
    x = np.asarray(x)
    out = []
    for i in indices:
        if i <= 0 or i >= len(x) - 1:
            continue
        if mode == "max":
            if (x[i] - max(x[i-1], x[i+1])) >= prominence:
                out.append(i)
        else:
            if (min(x[i-1], x[i+1]) - x[i]) >= prominence:
                out.append(i)
    return np.array(out, dtype=int)

In [5]:
# ---------- leitura do CSV e escolha do lado ----------
def pick_side(frames_df):
    """
    Escolhe LEFT ou RIGHT baseado em visibilidade média dos landmarks de braço.
    """
    left_vis = []
    right_vis = []
    # ombro, cotovelo, punho
    for lm in [11, 13, 15]:
        left_vis.append(frames_df.get(f"lm{lm}_vis"))
    for lm in [12, 14, 16]:
        right_vis.append(frames_df.get(f"lm{lm}_vis"))

    left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
    right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)

    left_score = np.nanmean(left)
    right_score = np.nanmean(right)
    return "LEFT" if left_score >= right_score else "RIGHT"


def get_arm_points(frames_df, side="LEFT", use_xy=True):
    """
    Retorna arrays A=ombro, B=cotovelo, C=punho.
    use_xy=True -> usa x,y (mais robusto pra câmera 2D)
    """
    if side == "LEFT":
        sh, el, wr = 11, 13, 15
    else:
        sh, el, wr = 12, 14, 16

    if use_xy:
        A = frames_df[[f"lm{sh}_x", f"lm{sh}_y"]].to_numpy()
        B = frames_df[[f"lm{el}_x", f"lm{el}_y"]].to_numpy()
        C = frames_df[[f"lm{wr}_x", f"lm{wr}_y"]].to_numpy()
    else:
        A = frames_df[[f"lm{sh}_x", f"lm{sh}_y", f"lm{sh}_z"]].to_numpy()
        B = frames_df[[f"lm{el}_x", f"lm{el}_y", f"lm{el}_z"]].to_numpy()
        C = frames_df[[f"lm{wr}_x", f"lm{wr}_y", f"lm{wr}_z"]].to_numpy()

    vis = frames_df[[f"lm{sh}_vis", f"lm{el}_vis", f"lm{wr}_vis"]].to_numpy()
    vis_mean = np.nanmean(vis, axis=1)
    return A, B, C, vis_mean


def interpolate_small_gaps(frames_df, cols):
    """
    Interpola NaNs em colunas numéricas (bom para pequenos buracos de detecção).
    """
    df = frames_df.copy()
    for c in cols:
        df[c] = df[c].interpolate(limit_direction="both")
    return df


## Métricas biomecânicas adicionais — VBT, aceleração, sticking point e ROM

Esta versão expande a extração por repetição para contemplar indicadores cinemáticos associados à fadiga muscular na rosca bíceps. Além das métricas originais, o `reps_df` passa a incluir velocidade angular concêntrica e excêntrica, perda percentual de velocidade ao longo da série, aceleração angular média e pico, indicadores de sticking point, deterioração da amplitude de movimento e métricas simples de fluidez/estabilidade articular.

Assumiu-se, conforme a detecção atual do notebook, que cada repetição segue o padrão `start -> peak -> end`, em que `start` e `end` correspondem a ângulos maiores do cotovelo, isto é, braço mais estendido, e `peak` corresponde ao menor ângulo, isto é, flexão máxima. Assim, a fase concêntrica é aproximada por `start -> peak` e a fase excêntrica por `peak -> end`.


In [6]:
# ---------- pipeline: frames -> reps + métricas biomecânicas avançadas ----------
def safe_div(num, den, default=np.nan):
    """Divisão segura para evitar inf/erro quando o denominador é zero ou NaN."""
    try:
        if den is None or not np.isfinite(den) or abs(den) < 1e-12:
            return default
        return float(num / den)
    except Exception:
        return default


def nan_stat(x, fn, default=np.nan):
    """Aplica uma estatística ignorando NaN e retorna default quando o vetor está vazio."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return default
    return float(fn(x))


def robust_gradient_by_time(y, t, fallback_dt=1/30):
    """
    Derivada numérica robusta para séries temporais.
    Retorna dy/dt mesmo quando o tempo possui valores ausentes ou não monotônicos.
    """
    y = np.asarray(y, dtype=float)
    t = np.asarray(t, dtype=float)

    if len(y) < 2:
        return np.zeros_like(y, dtype=float)

    dt = fallback_dt if np.isfinite(fallback_dt) and fallback_dt > 0 else 1/30
    valid_time = np.all(np.isfinite(t)) and np.all(np.diff(t) > 0)

    if valid_time:
        return np.gradient(y, t)
    return np.gradient(y, dt)


def trajectory_path_length(points):
    """Comprimento do caminho percorrido por uma articulação em coordenadas normalizadas."""
    points = np.asarray(points, dtype=float)
    if len(points) < 2:
        return np.nan
    diffs = np.diff(points, axis=0)
    dists = np.linalg.norm(diffs, axis=1)
    return nan_stat(dists, np.nansum)


def trajectory_displacement(points):
    """Deslocamento direto entre o primeiro e o último ponto de uma trajetória."""
    points = np.asarray(points, dtype=float)
    if len(points) < 2:
        return np.nan
    return float(np.linalg.norm(points[-1] - points[0]))


def count_velocity_reversals(omega_segment, min_speed=5.0):
    """
    Conta inversões relevantes de direção da velocidade angular.
    Ajuda a capturar movimento fragmentado/irregular, ignorando microoscilações.
    """
    omega_segment = np.asarray(omega_segment, dtype=float)
    omega_segment = omega_segment[np.isfinite(omega_segment)]
    if len(omega_segment) < 3:
        return 0

    signs = np.sign(omega_segment)
    signs[np.abs(omega_segment) < min_speed] = 0
    signs = signs[signs != 0]

    if len(signs) < 2:
        return 0
    return int(np.sum(signs[1:] != signs[:-1]))


def sticking_point_metrics(
    elbow_s,
    omega,
    t,
    start,
    peak,
    rom,
    speed_ratio=0.25,
    min_duration_sec=0.08,
    midrom_low=0.30,
    midrom_high=0.75,
):
    """
    Estima métricas de sticking point na fase concêntrica.

    No notebook, a fase concêntrica é assumida como start -> peak,
    isto é, do cotovelo mais estendido para a flexão máxima.
    O sticking point é aproximado como uma queda local acentuada da velocidade
    angular dentro da região intermediária da amplitude de movimento.
    """
    default = {
        "sticking_min_speed_deg_s": np.nan,
        "sticking_angle_deg": np.nan,
        "sticking_time_from_rep_start_sec": np.nan,
        "sticking_duration_sec": 0.0,
        "sticking_ratio_concentric": 0.0,
        "has_sticking_point": 0,
    }

    if peak <= start + 1 or not np.isfinite(rom) or rom <= 0:
        return default

    idx = np.arange(start, peak + 1)
    angles = np.asarray(elbow_s[idx], dtype=float)
    speed = np.abs(np.asarray(omega[idx], dtype=float))
    times = np.asarray(t[idx], dtype=float)

    valid = np.isfinite(angles) & np.isfinite(speed) & np.isfinite(times)
    if valid.sum() < 3:
        return default

    idx = idx[valid]
    angles = angles[valid]
    speed = speed[valid]
    times = times[valid]

    # Progresso dentro do ROM: 0 = início estendido; 1 = flexão máxima.
    rom_progress = (float(elbow_s[start]) - angles) / (rom + 1e-9)
    mid_mask = (rom_progress >= midrom_low) & (rom_progress <= midrom_high)

    # Caso a região intermediária fique curta por ruído de detecção, usa a fase concêntrica toda.
    if mid_mask.sum() < 3:
        mid_mask = np.ones_like(speed, dtype=bool)

    speed_mid = speed[mid_mask]
    times_mid = times[mid_mask]
    angles_mid = angles[mid_mask]

    baseline_speed = nan_stat(speed_mid, lambda v: np.nanpercentile(v, 75))
    if not np.isfinite(baseline_speed) or baseline_speed <= 0:
        return default

    threshold = speed_ratio * baseline_speed
    low_speed_mask = speed_mid <= threshold

    median_dt = nan_stat(np.diff(times), np.nanmedian, default=0.0)
    sticking_duration = float(np.sum(low_speed_mask) * median_dt) if np.isfinite(median_dt) else 0.0
    concentric_duration = float(t[peak] - t[start]) if np.isfinite(t[peak] - t[start]) else np.nan
    sticking_ratio = safe_div(sticking_duration, concentric_duration, default=0.0)

    local_min_pos = int(np.nanargmin(speed_mid))
    min_speed = float(speed_mid[local_min_pos])
    sticking_angle = float(angles_mid[local_min_pos])
    sticking_time_from_start = float(times_mid[local_min_pos] - t[start])
    has_sticking = int((sticking_duration >= min_duration_sec) and (min_speed <= threshold))

    return {
        "sticking_min_speed_deg_s": min_speed,
        "sticking_angle_deg": sticking_angle,
        "sticking_time_from_rep_start_sec": sticking_time_from_start,
        "sticking_duration_sec": sticking_duration,
        "sticking_ratio_concentric": sticking_ratio,
        "has_sticking_point": has_sticking,
    }


def add_series_progression_features(reps_df):
    """
    Adiciona métricas relativas ao avanço da série, como perda de velocidade,
    aumento de duração e deterioração do ROM em relação às primeiras repetições.
    """
    if reps_df is None or reps_df.empty:
        return reps_df

    reps_df = reps_df.copy()
    reps_df["rep_number_in_series"] = np.arange(1, len(reps_df) + 1)
    if len(reps_df) > 1:
        reps_df["series_progress_norm"] = (reps_df["rep_number_in_series"] - 1) / (len(reps_df) - 1)
    else:
        reps_df["series_progress_norm"] = 0.0

    def first_finite(series):
        vals = pd.to_numeric(series, errors="coerce").dropna()
        return float(vals.iloc[0]) if len(vals) else np.nan

    def slope_per_rep(series):
        y = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
        x = reps_df["rep_number_in_series"].to_numpy(dtype=float)
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 2:
            return np.nan
        return float(np.polyfit(x[mask], y[mask], 1)[0])

    # VBT: perda de velocidade principalmente na fase concêntrica.
    velocity_cols = {
        "concentric_velocity_mean_deg_s": "vbt_concentric_velocity",
        "rep_velocity_rom_per_sec_deg_s": "vbt_rep_velocity",
        "rep_velocity_path_deg_s": "vbt_path_velocity",
    }

    for col, prefix in velocity_cols.items():
        if col not in reps_df.columns:
            continue

        vals = pd.to_numeric(reps_df[col], errors="coerce")
        first_val = first_finite(vals)
        prev_vals = vals.shift(1)
        running_best = vals.cummax()

        reps_df[f"{prefix}_loss_from_first_pct"] = (first_val - vals) / (first_val + 1e-9) * 100
        reps_df[f"{prefix}_loss_from_previous_pct"] = (prev_vals - vals) / (prev_vals + 1e-9) * 100
        reps_df[f"{prefix}_loss_from_best_pct"] = (running_best - vals) / (running_best + 1e-9) * 100
        reps_df[f"{prefix}_slope_per_rep"] = slope_per_rep(vals)

    # Aumento de tempo da repetição e das fases.
    duration_cols = {
        "rep_duration": "rep_duration",
        "concentric_duration": "concentric_duration",
        "eccentric_duration": "eccentric_duration",
    }

    for col, prefix in duration_cols.items():
        if col not in reps_df.columns:
            continue
        vals = pd.to_numeric(reps_df[col], errors="coerce")
        first_val = first_finite(vals)
        prev_vals = vals.shift(1)
        reps_df[f"{prefix}_increase_from_first_pct"] = (vals - first_val) / (first_val + 1e-9) * 100
        reps_df[f"{prefix}_increase_from_previous_pct"] = (vals - prev_vals) / (prev_vals + 1e-9) * 100
        reps_df[f"{prefix}_slope_per_rep"] = slope_per_rep(vals)

    # Deterioração da amplitude de movimento e dos ângulos inicial/pico.
    if "rom_deg" in reps_df.columns:
        vals = pd.to_numeric(reps_df["rom_deg"], errors="coerce")
        first_val = first_finite(vals)
        running_best = vals.cummax()
        reps_df["rom_loss_from_first_pct"] = (first_val - vals) / (first_val + 1e-9) * 100
        reps_df["rom_loss_from_best_pct"] = (running_best - vals) / (running_best + 1e-9) * 100
        reps_df["rom_slope_deg_per_rep"] = slope_per_rep(vals)

    if "elbow_start_deg" in reps_df.columns:
        first_start = first_finite(reps_df["elbow_start_deg"])
        # Valor positivo sugere que a repetição começou com menor extensão do cotovelo.
        reps_df["elbow_start_extension_loss_from_first_deg"] = first_start - reps_df["elbow_start_deg"]
        reps_df["elbow_start_slope_deg_per_rep"] = slope_per_rep(reps_df["elbow_start_deg"])

    if "elbow_peak_deg" in reps_df.columns:
        first_peak = first_finite(reps_df["elbow_peak_deg"])
        # Valor positivo sugere menor flexão máxima em relação à primeira repetição.
        reps_df["elbow_peak_flexion_loss_from_first_deg"] = reps_df["elbow_peak_deg"] - first_peak
        reps_df["elbow_peak_slope_deg_per_rep"] = slope_per_rep(reps_df["elbow_peak_deg"])

    return reps_df


def build_reps_from_csv(
    csv_path,
    smooth_window=9,
    extrema_order=7,
    min_rep_seconds=0.6,
    min_rep_rom_deg=25.0,
    prominence_deg=3.0,
    sticking_speed_ratio=0.25,
    sticking_min_duration_sec=0.08,
    sticking_midrom_low=0.30,
    sticking_midrom_high=0.75,
):
    frames_df = pd.read_csv(csv_path)

    # garantir colunas essenciais
    if "frame" not in frames_df.columns or "time_sec" not in frames_df.columns:
        raise ValueError("CSV precisa ter colunas 'frame' e 'time_sec'.")

    # interpolar coordenadas básicas para reduzir NaNs
    coord_cols = [c for c in frames_df.columns if c.endswith(("_x", "_y", "_z"))]
    frames_df = interpolate_small_gaps(frames_df, coord_cols)

    # escolher lado automaticamente
    side = pick_side(frames_df)

    # obter pontos do braço
    A, B, C, vis_mean = get_arm_points(frames_df, side=side, use_xy=True)

    # ângulo do cotovelo por frame
    elbow = angle_deg(A, B, C)

    # suavização
    elbow_s = rolling_smooth(elbow, win=smooth_window)

    # fps aproximado
    t = frames_df["time_sec"].to_numpy(dtype=float)
    dt = np.nanmedian(np.diff(t))
    fps = 1.0 / dt if np.isfinite(dt) and dt > 0 else 30.0
    min_distance = int(round(min_rep_seconds * fps))

    # Derivadas temporais do ângulo do cotovelo.
    # omega < 0 geralmente indica fase concêntrica no curl detectado como max -> min.
    omega = robust_gradient_by_time(elbow_s, t, fallback_dt=1.0 / fps)
    alpha = robust_gradient_by_time(omega, t, fallback_dt=1.0 / fps)

    # extremos: no curl, geralmente:
    # - ângulo alto ~ braço estendido (topo do ângulo)
    # - ângulo baixo ~ braço flexionado (mínimo do ângulo)
    maxima = local_extrema(elbow_s, order=extrema_order, mode="max")
    minima = local_extrema(elbow_s, order=extrema_order, mode="min")

    # filtros básicos
    maxima = filter_by_distance(maxima, min_distance=min_distance)
    minima = filter_by_distance(minima, min_distance=min_distance)

    maxima = filter_by_prominence(elbow_s, maxima, prominence=prominence_deg, mode="max")
    minima = filter_by_prominence(elbow_s, minima, prominence=prominence_deg, mode="min")

    # montar reps: (max -> min -> max)
    reps = []

    print("SIDE:", side)
    print("elbow_s stats:",
        "min", float(np.nanmin(elbow_s)),
        "max", float(np.nanmax(elbow_s)),
        "range", float(np.nanmax(elbow_s) - np.nanmin(elbow_s)))

    # Quantos extremos antes de filtros?
    maxima0 = local_extrema(elbow_s, order=extrema_order, mode="max")
    minima0 = local_extrema(elbow_s, order=extrema_order, mode="min")
    print("extrema raw - maxima:", len(maxima0), "minima:", len(minima0))

    # Depois de distância
    maxima1 = filter_by_distance(maxima0, min_distance=int(round(min_rep_seconds * fps)))
    minima1 = filter_by_distance(minima0, min_distance=int(round(min_rep_seconds * fps)))
    print("extrema dist - maxima:", len(maxima1), "minima:", len(minima1))

    # Se você estiver usando prominence, veja o impacto
    maxima2 = filter_by_prominence(elbow_s, maxima1, prominence=prominence_deg, mode="max")
    minima2 = filter_by_prominence(elbow_s, minima1, prominence=prominence_deg, mode="min")
    print("extrema prom - maxima:", len(maxima2), "minima:", len(minima2))

    # estratégia: percorre máximos e busca um mínimo entre eles
    for i in range(len(maxima) - 1):
        start = maxima[i]
        end = maxima[i + 1]
        if end - start < min_distance:
            continue

        # mínimos dentro do intervalo
        mins_between = minima[(minima > start) & (minima < end)]
        if len(mins_between) == 0:
            continue

        # pega o mínimo mais “profundo” (menor ângulo) dentro do intervalo
        peak = mins_between[np.argmin(elbow_s[mins_between])]

        # ROM (diferença entre ângulo estendido e flexionado)
        rom = elbow_s[start] - elbow_s[peak]
        if np.isnan(rom) or rom < min_rep_rom_deg:
            continue

        start_time = float(frames_df.loc[start, "time_sec"])
        peak_time = float(frames_df.loc[peak, "time_sec"])
        end_time = float(frames_df.loc[end, "time_sec"])

        rep_duration = end_time - start_time
        concentric_duration = peak_time - start_time
        eccentric_duration = end_time - peak_time

        rep_slice = slice(start, end + 1)
        conc_slice = slice(start, peak + 1)
        ecc_slice = slice(peak, end + 1)

        rep_speed = np.abs(omega[rep_slice])
        conc_speed = np.abs(omega[conc_slice])
        ecc_speed = np.abs(omega[ecc_slice])

        rep_accel_abs = np.abs(alpha[rep_slice])
        conc_accel_abs = np.abs(alpha[conc_slice])
        ecc_accel_abs = np.abs(alpha[ecc_slice])

        angular_path = nan_stat(np.abs(np.diff(elbow_s[rep_slice])), np.nansum)
        conc_angular_path = nan_stat(np.abs(np.diff(elbow_s[conc_slice])), np.nansum)
        ecc_angular_path = nan_stat(np.abs(np.diff(elbow_s[ecc_slice])), np.nansum)

        sticking = sticking_point_metrics(
            elbow_s=elbow_s,
            omega=omega,
            t=t,
            start=start,
            peak=peak,
            rom=rom,
            speed_ratio=sticking_speed_ratio,
            min_duration_sec=sticking_min_duration_sec,
            midrom_low=sticking_midrom_low,
            midrom_high=sticking_midrom_high,
        )

        shoulder_traj = A[rep_slice]
        elbow_traj = B[rep_slice]
        wrist_traj = C[rep_slice]
        wrist_path = trajectory_path_length(wrist_traj)
        wrist_disp = trajectory_displacement(wrist_traj)

        velocity_reversal_count = count_velocity_reversals(omega[rep_slice], min_speed=5.0)

        reps.append({
            "rep_id": len(reps),
            "side": side,
            "start_idx": int(start),
            "peak_idx": int(peak),   # flexão máxima (menor ângulo)
            "end_idx": int(end),
            "start_frame": int(frames_df.loc[start, "frame"]),
            "peak_frame": int(frames_df.loc[peak, "frame"]),
            "end_frame": int(frames_df.loc[end, "frame"]),
            "start_time": start_time,
            "peak_time": peak_time,
            "end_time": end_time,
            "rep_duration": float(rep_duration),
            "concentric_duration": float(concentric_duration),
            "eccentric_duration": float(eccentric_duration),
            "rom_deg": float(rom),
            "elbow_start_deg": float(elbow_s[start]),
            "elbow_peak_deg": float(elbow_s[peak]),
            "elbow_end_deg": float(elbow_s[end]),
            "mean_vis_arm": float(np.nanmean(vis_mean[start:end+1])),

            # VBT / velocidade angular média por fase
            "concentric_velocity_mean_deg_s": safe_div(rom, concentric_duration),
            "eccentric_velocity_mean_deg_s": safe_div(abs(elbow_s[end] - elbow_s[peak]), eccentric_duration),
            "rep_velocity_rom_per_sec_deg_s": safe_div(rom, rep_duration),
            "rep_velocity_path_deg_s": safe_div(angular_path, rep_duration),
            "concentric_path_velocity_deg_s": safe_div(conc_angular_path, concentric_duration),
            "eccentric_path_velocity_deg_s": safe_div(ecc_angular_path, eccentric_duration),

            # Velocidade instantânea: pico e dispersão
            "concentric_speed_peak_deg_s": nan_stat(conc_speed, np.nanmax),
            "eccentric_speed_peak_deg_s": nan_stat(ecc_speed, np.nanmax),
            "rep_speed_peak_deg_s": nan_stat(rep_speed, np.nanmax),
            "concentric_speed_mean_abs_deg_s": nan_stat(conc_speed, np.nanmean),
            "eccentric_speed_mean_abs_deg_s": nan_stat(ecc_speed, np.nanmean),
            "rep_speed_mean_abs_deg_s": nan_stat(rep_speed, np.nanmean),
            "concentric_velocity_cv": safe_div(nan_stat(conc_speed, np.nanstd), nan_stat(conc_speed, np.nanmean)),
            "rep_velocity_cv": safe_div(nan_stat(rep_speed, np.nanstd), nan_stat(rep_speed, np.nanmean)),

            # Aceleração angular média e pico
            "concentric_accel_mean_abs_deg_s2": nan_stat(conc_accel_abs, np.nanmean),
            "eccentric_accel_mean_abs_deg_s2": nan_stat(ecc_accel_abs, np.nanmean),
            "rep_accel_mean_abs_deg_s2": nan_stat(rep_accel_abs, np.nanmean),
            "concentric_accel_peak_abs_deg_s2": nan_stat(conc_accel_abs, np.nanmax),
            "eccentric_accel_peak_abs_deg_s2": nan_stat(ecc_accel_abs, np.nanmax),
            "rep_accel_peak_abs_deg_s2": nan_stat(rep_accel_abs, np.nanmax),

            # Fluidez / fragmentação do movimento
            "velocity_reversal_count": velocity_reversal_count,
            "movement_fragmentation_index": safe_div(velocity_reversal_count, rep_duration, default=0.0),

            # Sticking point
            **sticking,

            # Estabilidade/compensação articular em coordenadas normalizadas do MediaPipe
            "shoulder_path_length_norm": trajectory_path_length(shoulder_traj),
            "elbow_path_length_norm": trajectory_path_length(elbow_traj),
            "wrist_path_length_norm": wrist_path,
            "shoulder_displacement_norm": trajectory_displacement(shoulder_traj),
            "elbow_displacement_norm": trajectory_displacement(elbow_traj),
            "wrist_displacement_norm": wrist_disp,
            "wrist_path_efficiency": safe_div(wrist_disp, wrist_path),
        })

    reps_df = pd.DataFrame(reps)
    reps_df = add_series_progression_features(reps_df)

    # também devolve o frames_df com sinais guia (útil para plot/depurar)
    frames_out = frames_df.copy()
    frames_out["elbow_deg_raw"] = elbow
    frames_out["elbow_deg_smooth"] = elbow_s
    frames_out["elbow_angular_velocity_deg_s"] = omega
    frames_out["elbow_angular_speed_abs_deg_s"] = np.abs(omega)
    frames_out["elbow_angular_acceleration_deg_s2"] = alpha
    frames_out["arm_vis_mean"] = vis_mean

    return frames_out, reps_df


In [7]:
# ---------- exemplo de uso ----------
if __name__ == "__main__":
    raiz = r"./saida/landmarks/"
    saida = raiz.replace("landmarks/", "feature_extraction/")
    os.makedirs(saida, exist_ok=True)

    arquivos = listar_arquivos(raiz) or []
    for arquivo in arquivos:
        csv_path = os.path.join(raiz, arquivo)
        print(csv_path)
        frames_df, reps_df = build_reps_from_csv(
            csv_path,
            smooth_window=11,
            extrema_order=7,
            min_rep_seconds=0.7,
            min_rep_rom_deg=25.0,
            prominence_deg=0,
        )

        print("Reps detectadas:", len(reps_df))
        cols_preview = [
            "rep_id",
            "rep_duration",
            "concentric_duration",
            "eccentric_duration",
            "rom_deg",
            "concentric_velocity_mean_deg_s",
            "vbt_concentric_velocity_loss_from_first_pct",
            "concentric_accel_peak_abs_deg_s2",
            "sticking_min_speed_deg_s",
            "has_sticking_point",
            "rom_loss_from_first_pct",
        ]
        cols_preview = [c for c in cols_preview if c in reps_df.columns]
        print(reps_df[cols_preview].head())

        # Salvar para revisar/corrigir manualmente se quiser
        output_path = os.path.join(saida, arquivo.replace("_landmarks.csv", "_reps.csv"))
        reps_df.to_csv(output_path, index=False)
        print("Arquivo salvo em:", output_path)


./saida/landmarks/aug_brightness_friends_record_01_landmarks.csv
SIDE: RIGHT
elbow_s stats: min 4.08136569936849 max 178.826004904449 range 174.74463920508052
extrema raw - maxima: 61 minima: 58
extrema dist - maxima: 50 minima: 46
extrema prom - maxima: 50 minima: 46
Reps detectadas: 36
   rep_id  rep_duration  concentric_duration  eccentric_duration     rom_deg  \
0       0      0.733333             0.600000            0.133333   54.370219   
1       1      1.400000             0.800000            0.600000  155.541512   
2       2      0.966667             0.266667            0.700000   35.499726   
3       3      0.933333             0.233333            0.700000   64.519755   
4       4      1.533333             1.400000            0.133333  112.351931   

   concentric_velocity_mean_deg_s  \
0                       90.617032   
1                      194.426890   
2                      133.123974   
3                      276.513237   
4                       80.251379   

   vbt_

C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)


SIDE: LEFT
elbow_s stats: min 8.178276387216373 max 175.1656865785953 range 166.98741019137893
extrema raw - maxima: 19 minima: 20
extrema dist - maxima: 19 minima: 20
extrema prom - maxima: 19 minima: 20
Reps detectadas: 18
   rep_id  rep_duration  concentric_duration  eccentric_duration     rom_deg  \
0       0      2.033333             0.866667            1.166667  135.080999   
1       1      2.333333             0.966667            1.366667  143.827436   
2       2      2.133333             1.033333            1.100000  148.803696   
3       3      2.166667             1.000000            1.166667  133.731661   
4       4      2.300000             1.100000            1.200000  148.678809   

   concentric_velocity_mean_deg_s  \
0                      155.862691   
1                      148.787003   
2                      144.003576   
3                      133.731661   
4                      135.162554   

   vbt_concentric_velocity_loss_from_first_pct  \
0                    

C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)


SIDE: RIGHT
elbow_s stats: min 3.599042537021928 max 177.1072641614517 range 173.50822162442978
extrema raw - maxima: 9 minima: 10
extrema dist - maxima: 9 minima: 9
extrema prom - maxima: 9 minima: 9
Reps detectadas: 8
   rep_id  rep_duration  concentric_duration  eccentric_duration     rom_deg  \
0       0      1.966667             0.933333            1.033333  163.235523   
1       1      1.700000             0.733333            0.966667  157.335455   
2       2      1.666667             0.733333            0.933333  164.202849   
3       3      1.666667             0.600000            1.066667  153.455817   
4       4      1.766667             1.000000            0.766667  159.485737   

   concentric_velocity_mean_deg_s  \
0                      174.895203   
1                      214.548347   
2                      223.912975   
3                      255.759695   
4                      159.485737   

   vbt_concentric_velocity_loss_from_first_pct  \
0                         

...

C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)


SIDE: LEFT
elbow_s stats: min 7.5630729964181285 max 122.62325027657394 range 115.06017728015581
extrema raw - maxima: 7 minima: 8
extrema dist - maxima: 7 minima: 8
extrema prom - maxima: 7 minima: 8
Reps detectadas: 6
   rep_id  rep_duration  concentric_duration  eccentric_duration     rom_deg  \
0       0      2.000000             1.033333            0.966667  104.703990   
1       1      2.133333             1.100000            1.033333  101.543010   
2       2      2.100000             1.100000            1.000000   98.417331   
3       3      2.333333             1.233333            1.100000  107.834106   
4       4      2.466667             1.333333            1.133333  103.432477   

   concentric_velocity_mean_deg_s  \
0                      101.326442   
1                       92.311827   
2                       89.470301   
3                       87.433059   
4                       77.574358   

   vbt_concentric_velocity_loss_from_first_pct  \
0                         

C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)


SIDE: LEFT
elbow_s stats: min 2.8170852294393987 max 171.78230997034333 range 168.96522474090395
extrema raw - maxima: 6 minima: 7
extrema dist - maxima: 5 minima: 4
extrema prom - maxima: 5 minima: 4
Reps detectadas: 3
   rep_id  rep_duration  concentric_duration  eccentric_duration     rom_deg  \
0       0      3.633333             1.666667            1.966667  167.011470   
1       1      2.566667             2.233333            0.333333  164.227807   
2       2      2.433333             2.100000            0.333333  163.544093   

   concentric_velocity_mean_deg_s  \
0                      100.206882   
1                       73.534839   
2                       77.878140   

   vbt_concentric_velocity_loss_from_first_pct  \
0                                     0.000000   
1                                    26.616977   
2                                    22.282644   

   concentric_accel_peak_abs_deg_s2  sticking_min_speed_deg_s  \
0                       1844.912273         

C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_47336\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)
